In [1]:
# Import the clean_sac_file function from the previous example
from obspy import read, Trace, Stream
from obspy.signal.filter import bandpass
import numpy as np
import os

directory = os.getcwd()
directory = directory[len(directory) - 6:]

# Define the directory path
dir_path = './'
orderdata = directory + "_"

os.makedirs("class0") # yok
os.makedirs("class0/clean") 
os.makedirs("class0/original")

os.makedirs("class1") # 1 seviye 250
os.makedirs("class1/clean")
os.makedirs("class1/original")

os.makedirs("class2") # 2 seviye 500
os.makedirs("class2/clean")
os.makedirs("class2/original")

os.makedirs("class3") # 3 seviye 1000
os.makedirs("class3/clean")
os.makedirs("class3/original")

os.makedirs("class4") # 4 seviye 2000
os.makedirs("class4/clean")
os.makedirs("class4/original")

os.makedirs("class5") # 5 seviye 4000
os.makedirs("class5/clean")
os.makedirs("class5/original")

os.makedirs("class6") # 6 seviye 8000   
os.makedirs("class6/clean")
os.makedirs("class6/original")

os.makedirs("class7") # 6 seviye 16000   
os.makedirs("class7/clean")
os.makedirs("class7/original")

os.makedirs("class8") # 6 seviye 32000   
os.makedirs("class8/clean")
os.makedirs("class8/original")




def clean_sac_file(input_file, output_file, freq_min = 2.0, freq_max = 4.3, time_pad = 10.001):
    """
    Applies a bandpass filter to a SAC file to clean noise.
    Inputs:
    - input_file: path to the input SAC file
    - output_file: path to the output SAC file
    - freq_min: minimum frequency of the passband in Hz
    - freq_max: maximum frequency of the passband in Hz
    """
    # Read in the SAC file as an ObsPy Trace object
    trace = read(input_file)[0]
    # trace.plot()
    
    # Apply a bandpass filter to the trace
    trace_filtered = trace.copy()
    t = trace_filtered.stats.starttime

    if t + time_pad > trace_filtered.stats.endtime:
        return None

    if len(trace_filtered.data) < 2000:
        return None
    
    trace_filtered = trace_filtered.trim(t + time_pad, trace_filtered.stats.endtime) 
    # trace_filtered = trace_filtered.trim(t, trace_filtered.stats.endtime)
    # trace_filtered.data = np.concatenate((np.zeros(750), trace_filtered.data))
    
    trace_filtered.data = bandpass(trace_filtered.data, freqmin=freq_min, freqmax=freq_max, df=trace_filtered.stats.sampling_rate)
    # trace_filtered = trace_filtered.trim(t + time_pad, trace_filtered.stats.endtime)
    # trace_filtered.plot()

    return trace_filtered

/usr/lib/python3/dist-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 1.1build1 is an invalid version and will not be supported in a future release
  warnings.warn(
/usr/lib/python3/dist-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.43ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(


In [2]:
import os

#to get the current working directory


In [3]:
import shutil
from obspy import read, Trace, Stream

clean_files = []

for i, folder_month in enumerate(os.listdir(dir_path)):

    if not os.path.isfile(dir_path + folder_month):
        # print(folder_month)

        if folder_month.__contains__("class"):
            continue

        for j, eartq in enumerate(os.listdir(dir_path + folder_month)):

            if not os.path.join(dir_path, folder_month, eartq).format().__contains__("BHZ"):
                continue

            # print(f"    {eartq}")
            
            date = str(i) + "_" + str(j) + "_"
            
            # c = c + 1
            # continue

            # Define the input and output file paths
            input_file = dir_path + folder_month + '/' + eartq
            output_file = orderdata + date + eartq.replace('(', '').replace(')', '').replace('=', '').replace('KO', 'SAC')


            file_name = dir_path + folder_month + '/' + eartq


            if clean_files.__contains__(file_name):
                continue
            
            print(input_file)
            trace_filtered = clean_sac_file(input_file, output_file)
            if trace_filtered is None:
                continue

            clean_files.append(file_name)

            # if trace_filtered.stast.endtime - trace_filtered.stats.starttime < 15:
            #     continue

            trace_filtered = trace_filtered[300:]
            if len(trace_filtered.data) < 2000:
                continue

            # crop 8000 sample in the middle
            lengthOfArray = len(trace_filtered.data)

            max_val = max(trace_filtered.data)
            max_value_index = np.argmax(trace_filtered.data)

            min_val = min(trace_filtered.data)
            min_value_index = np.argmin(trace_filtered.data)

            write_signal = read(input_file)[0]
            max_orig_val = max(write_signal.data)
            min_orig_val = min(write_signal.data)

            add_num = 0
            # according to max and min value values place the array in the middle by 0 value
            # if max_orig_val < 0 and min_orig_val < 0:
            #     diff = abs(max_orig_val) - abs(min_orig_val)
            #     add_num = diff
            # elif max_orig_val > 0 and min_orig_val > 0:
            #     diff = abs(max_orig_val) - abs(min_orig_val)
            #     add_num = -diff

            a = 0
            res_signal = Trace()
            if lengthOfArray - max_value_index < 1000:
                a = 1
                rightArr = trace_filtered.data[max_value_index:]
                rest_length = 2000 - len(rightArr)
                leftArr = trace_filtered.data[max_value_index - rest_length: max_value_index]
                res_signal.data = np.concatenate((leftArr, rightArr), axis=0)

                write_signal.data = write_signal.data[max_value_index - rest_length:] + add_num

            elif max_value_index < 1000:
                a = 2
                leftArr = trace_filtered.data[0: max_value_index]
                rest_length = 2000 - len(leftArr)
                rightArr = trace_filtered.data[max_value_index: max_value_index + rest_length]
                res_signal.data = np.concatenate((leftArr, rightArr), axis=0)

                write_signal.data = write_signal.data[0: max_value_index + rest_length] + add_num

            else:
                a = 3
                res_signal.data = np.array(trace_filtered.data[max_value_index - 1000: max_value_index + 1000])

                write_signal.data = write_signal.data[max_value_index - 1000: max_value_index + 1000] + add_num

            if len(res_signal.data) < 2000:
                print("error" + (str(a)))
                continue

            x3 = 1
            x2 = 3
            x1 = 16

            print(f"Len : {len(write_signal.data)}")

            random_number = np.random.randint(0, x1 + x2 + x3)
            
            if random_number < x1:
                max_val = max_val * 3
            elif random_number < x1 + x2:
                max_val = max_val * 2
            else:
                max_val = max_val * 1


            if max_val < 250:
                res_signal.write('./class0/clean/' + output_file, format='SAC')
                write_signal.write('./class0/original/' + orderdata + date + eartq + '_original.sac', format='SAC')
            elif max_val < 500:
                res_signal.write('./class1/clean/' + output_file, format='SAC')
                write_signal.write('./class1/original/' + orderdata + date + eartq + '_original.sac', format='SAC')
            elif max_val < 1000:
                res_signal.write('./class2/clean/' + output_file, format='SAC')
                write_signal.write('./class2/original/' + orderdata + date + eartq + '_original.sac', format='SAC')
            elif max_val < 2000:
                res_signal.write('./class3/clean/' + output_file, format='SAC')
                write_signal.write('./class3/original/' + orderdata + date + eartq + '_original.sac', format='SAC')
            elif max_val < 4000:
                res_signal.write('./class4/clean/' + output_file, format='SAC')
                write_signal.write('./class4/original/' + orderdata + date + eartq + '_original.sac', format='SAC')
            elif max_val < 8000:
                res_signal.write('./class5/clean/' + output_file, format='SAC')
                write_signal.write('./class5/original/' + orderdata + date + eartq + '_original.sac', format='SAC')
            elif max_val < 16000:
                res_signal.write('./class6/clean/' + output_file, format='SAC')
                write_signal.write('./class6/original/' + orderdata + date + eartq + '_original.sac', format='SAC')
            elif max_val < 32000:
                res_signal.write('./class7/clean/' + output_file, format='SAC')
                write_signal.write('./class7/original/' + orderdata + date + eartq + '_original.sac', format='SAC')
            else:
                res_signal.write('./class8/clean/' + output_file, format='SAC')
                write_signal.write('./class8/original/' + orderdata + date + eartq + '_original.sac', format='SAC')           


./20211114_15193239.13-29.07Ke-SOGUT-SIMAV-KUTAHYA.M=2.0/SIMA.BHZ.KO
Len : 2000
./20211114_15193239.13-29.07Ke-SOGUT-SIMAV-KUTAHYA.M=2.0/GEML.BHZ.KO
Len : 2000
./20211114_15193239.13-29.07Ke-SOGUT-SIMAV-KUTAHYA.M=2.0/CAVI.BHZ.KO
Len : 2000
./20211114_15193239.13-29.07Ke-SOGUT-SIMAV-KUTAHYA.M=2.0/GEDZ.BHZ.KO
Len : 2000
./20211114_15193239.13-29.07Ke-SOGUT-SIMAV-KUTAHYA.M=2.0/SDAG.BHZ.KO
Len : 2000
./20211114_15193239.13-29.07Ke-SOGUT-SIMAV-KUTAHYA.M=2.0/SVRH.BHZ.KO
Len : 2000
./20211114_15193239.13-29.07Ke-SOGUT-SIMAV-KUTAHYA.M=2.0/ADVT.BHZ.KO
Len : 2000
./20211114_15193239.13-29.07Ke-SOGUT-SIMAV-KUTAHYA.M=2.0/GULT.BHZ.KO
Len : 2000
./20211114_15193239.10-29.05Ke-GOKCELER-SIMAV-KUTAHYA.M=1.7/BALB.BHZ.KO
Len : 2000
./20211114_15193239.10-29.05Ke-GOKCELER-SIMAV-KUTAHYA.M=1.7/SIMA.BHZ.KO
Len : 2000
./20211114_15193239.10-29.05Ke-GOKCELER-SIMAV-KUTAHYA.M=1.7/GEML.BHZ.KO
Len : 2000
./20211114_15193239.10-29.05Ke-GOKCELER-SIMAV-KUTAHYA.M=1.7/CAVI.BHZ.KO
Len : 2000
./20211114_15193239.10-29.05